# Sorcha vs benchmark: dissecting the 110 matched pairs

**What are the 110 pairs?**
- Filter Sorcha to tracklets with `|mjd0_utc − 61041.0| ≤ 1 day` (i.e., observed within ±1 day of 2026-01-01, the benchmark's fixed epoch)
- For each, read the S3M `.s3m` files to get the orbital elements `(e, H)` of that object
- KDTree nearest-neighbour match in `(e, H)` space to the benchmark; keep matches with distance < 1e-4 (exact same physical object)
- Result: 110 pairs — same S3M asteroid, observed by Sorcha near MJD 61041 AND in the synthetic benchmark at exactly MJD 61041

**No direction filter** — pairs come from wherever LSST pointed on those two nights.

In [ ]:
import glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
%matplotlib inline

MAP_EPOCH_MJD = 61041.0

df = pd.read_parquet("outputs/phase2_s3m/sorcha_comparison_s3m.parquet")
df["absdlon"] = df.prob_map_file.map(
    lambda s: (lambda m: abs(int(m.group(1))) if m else 999)(re.search(r"dlon([+-]?\d+)", str(s))))

dfb_all = pd.read_parquet("outputs/phase2_benchmark_s3m/benchmark_comparison_s3m.parquet")
dfb = dfb_all[dfb_all.mag_bin_label.notna()].copy().reset_index(drop=True)
dfb["absdlon"] = dfb.dlon_from_antisun_deg.abs()

# Read S3M orbital elements (e, H) for each ObjID
s3m_records = []
for f in sorted(glob.glob("S*.s3m")):
    with open(f) as fh:
        for line in fh:
            if line.startswith("!"): continue
            parts = line.split()
            if len(parts) < 9: continue
            try: s3m_records.append((parts[0], float(parts[3]), float(parts[8])))
            except ValueError: pass
s3m_eh = (pd.DataFrame(s3m_records, columns=["ObjID", "e_s3m", "H_s3m"])
          .drop_duplicates("ObjID").set_index("ObjID"))

# Epoch filter: Sorcha tracklets within ±1d of map epoch
near = df[(df.mjd0_utc - MAP_EPOCH_MJD).abs() <= 1].copy()
near = near.join(s3m_eh, on="ObjID").dropna(subset=["e_s3m", "H_s3m"])

# KDTree match on (e, H)
tree = cKDTree(dfb[["e", "H"]].values)
dist, idx = tree.query(near[["e_s3m", "H_s3m"]].values, k=1)
near = near.copy()
near["match_dist"] = dist
near["bench_idx"] = idx
matched = near[near.match_dist < 1e-4].copy().reset_index(drop=True)
bench   = dfb.iloc[matched.bench_idx.values].copy().reset_index(drop=True)

print(f"Sorcha tracklets within ±1d of MJD {MAP_EPOCH_MJD}: {len(near):,}")
print(f"Confident same-object matches:          {len(matched)}")
print(f"Sorcha absdlon range: {matched.absdlon.min():.0f}–{matched.absdlon.max():.0f}°")
print(f"Benchmark absdlon range: {bench.absdlon.min():.0f}–{bench.absdlon.max():.0f}°")
print()
print("Sorcha mjd0 range:", matched.mjd0_utc.min(), "–", matched.mjd0_utc.max())
print("Benchmark mjd0 range:", bench.mjd0_utc.min(), "–", bench.mjd0_utc.max())
print()
dv = np.hypot(matched.vlam.values - bench.vlam.values,
              matched.vbeta.values - bench.vbeta.values)
print(f"|dv| between matched pairs: median={np.median(dv):.4f}  mean={np.mean(dv):.4f}  max={dv.max():.4f} deg/day")
print(f"dt_min (Sorcha): median={matched.dt_min.median():.1f}  benchmark: always 30.0 min")

In [ ]:
# Side-by-side summary table (first 15 rows)
tbl = pd.DataFrame({
    "ObjID":        matched.ObjID.values,
    "S_mjd0":       matched.mjd0_utc.round(3).values,
    "S_absdlon":    matched.absdlon.values,
    "S_dt_min":     matched.dt_min.round(1).values,
    "S_vlam":       matched.vlam.round(4).values,
    "S_vbeta":      matched.vbeta.round(4).values,
    "B_mjd0":       bench.mjd0_utc.round(3).values,
    "B_absdlon":    bench.absdlon.round(1).values,
    "B_vlam":       bench.vlam.round(4).values,
    "B_vbeta":      bench.vbeta.round(4).values,
    "|dv|": dv.round(4),
})
print(tbl.head(15).to_string(index=False))
print(f"\n...{len(tbl)} rows total")

## Rate space: scatter plot (all 110 pairs)

Each point = one S3M object. Red = Sorcha observation (real LSST visit, noise, variable baseline). Blue = benchmark (synthetic, MJD 61041 exactly, 30-min baseline). Gray line connects the same object.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

for i in range(len(matched)):
    ax.plot([matched.vlam.iloc[i], bench.vlam.iloc[i]],
            [matched.vbeta.iloc[i], bench.vbeta.iloc[i]],
            color="gray", lw=0.5, alpha=0.4, zorder=1)

ax.scatter(bench.vlam,   bench.vbeta,   c="tab:blue", s=35, alpha=0.8, zorder=3,
           marker="^", label=f"Benchmark (MJD {MAP_EPOCH_MJD:.0f}, synthetic 30-min)")
ax.scatter(matched.vlam, matched.vbeta, c="tab:red",  s=35, alpha=0.8, zorder=4,
           label=f"Sorcha (±1d epoch, real LSST visits)")

ax.axhline(0, color="k", lw=0.4)
ax.axvline(0, color="k", lw=0.4)
ax.set_xlabel(r"$v_\lambda$ (deg/day)", fontsize=12)
ax.set_ylabel(r"$v_\beta$ (deg/day)", fontsize=12)
ax.set_title(f"Ecliptic rate space — {len(matched)} matched pairs\n"
             f"Same S3M object, two pipelines. Gray = same object.", fontsize=11)
ax.legend(fontsize=10)
ax.set_aspect("equal")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("Figures/matched_pairs_ratespace_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved Figures/matched_pairs_ratespace_scatter.png")

## Rate space: quiver arrows from origin (all 110 pairs)

Each arrow = velocity vector of one object. Both arrows for the same object start at the origin — so if Sorcha and benchmark agree perfectly, the two arrows would overlap.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

zeros = np.zeros(len(matched))
ax.quiver(zeros, zeros, bench.vlam,   bench.vbeta,   color="tab:blue", alpha=0.5,
          angles="xy", scale_units="xy", scale=1, width=0.003,
          label=f"Benchmark (MJD {MAP_EPOCH_MJD:.0f})")
ax.quiver(zeros, zeros, matched.vlam, matched.vbeta, color="tab:red",  alpha=0.5,
          angles="xy", scale_units="xy", scale=1, width=0.003,
          label="Sorcha (±1d epoch)")

lim = 1.1 * max(
    matched[["vlam", "vbeta"]].abs().values.max(),
    bench[["vlam", "vbeta"]].abs().values.max()
)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.axhline(0, color="k", lw=0.4)
ax.axvline(0, color="k", lw=0.4)
ax.set_xlabel(r"$v_\lambda$ (deg/day)", fontsize=12)
ax.set_ylabel(r"$v_\beta$ (deg/day)", fontsize=12)
ax.set_title(f"Velocity vectors from origin — {len(matched)} matched pairs\n"
             f"Red=Sorcha, Blue=benchmark. Overlap = agreement.", fontsize=11)
ax.legend(fontsize=10)
ax.set_aspect("equal")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("Figures/matched_pairs_quiver.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved Figures/matched_pairs_quiver.png")

## Where do the 110 Sorcha tracklets point? (direction plot)

The 110 Sorcha tracklets were observed within ±1d of MJD 61041 — but only in directions LSST actually pointed those nights. The benchmark has no such constraint: it covers all sky directions uniformly. This is the root cause of the performance gap in full-sky metrics.

In [ ]:
BINS = [(0, 20), (20, 40), (40, 70), (70, 110), (110, 141)]
bin_labels = [f"{lo}–{hi}°" for lo, hi in BINS]

sorcha_counts = [((matched.absdlon >= lo) & (matched.absdlon < hi)).sum() for lo, hi in BINS]
bench_counts  = [((bench.absdlon   >= lo) & (bench.absdlon   < hi)).sum() for lo, hi in BINS]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Panel 1: absdlon histogram
ax = axes[0]
xs = [0.5*(lo+hi) for lo, hi in BINS]
widths = [hi-lo for lo, hi in BINS]
ax.bar([x-3 for x in xs], [100*c/len(matched) for c in sorcha_counts],
       width=[w*0.4 for w in widths], color="tab:red",  alpha=0.7, label="Sorcha (N=110)")
ax.bar([x+3 for x in xs], [100*c/len(bench)   for c in bench_counts],
       width=[w*0.4 for w in widths], color="tab:blue", alpha=0.7, label="Benchmark (N=110)")
ax.set_xticks(xs); ax.set_xticklabels(bin_labels, fontsize=9)
ax.set_xlabel("|Δλ☉| direction bin")
ax.set_ylabel("% of 110 pairs")
ax.set_title("Direction bin distribution\n(same 110 objects, two pipelines)")
ax.legend()
ax.grid(alpha=0.2, axis="y")

print(f"{'Bin':>8}  {'Sorcha N':>9}  {'Bench N':>8}")
for (lo, hi), sc, bc in zip(BINS, sorcha_counts, bench_counts):
    print(f"{lo:>3}-{hi:<3}°   {sc:>9}  {bc:>8}")

# Panel 2: Sorcha sky positions (RA/Dec)
ax = axes[1]
sc = ax.scatter(matched.ra0, matched.dec0, c=matched.absdlon,
                cmap="plasma", s=30, alpha=0.8, vmin=0, vmax=141)
plt.colorbar(sc, ax=ax, label="|Δλ☉| (deg)")
ax.set_xlabel("RA (deg)"); ax.set_ylabel("Dec (deg)")
ax.set_title("Sorcha tracklet sky positions\n(colored by |Δλ☉|, ±1d of MJD 61041)")
ax.grid(alpha=0.2)

# Panel 3: Benchmark sky positions
ax = axes[2]
sc = ax.scatter(bench.ra0, bench.dec0, c=bench.absdlon,
                cmap="plasma", s=30, alpha=0.8, vmin=0, vmax=141, marker="^")
plt.colorbar(sc, ax=ax, label="|Δλ☉| (deg)")
ax.set_xlabel("RA (deg)"); ax.set_ylabel("Dec (deg)")
ax.set_title("Benchmark tracklet sky positions\n(same 110 objects, MJD 61041 exactly)")
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig("Figures/matched_pairs_directions.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved Figures/matched_pairs_directions.png")

## Rate space by direction bin — scatter (5 panels)

Sorcha absdlon used to assign direction bins. Same object may fall in a different bin in the benchmark (because the benchmark epoch is exactly MJD 61041 so the antisun direction is slightly different).

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 5), sharey=True)

for ax, (lo, hi) in zip(axes, BINS):
    ms = (matched.absdlon >= lo) & (matched.absdlon < hi)
    n = ms.sum()
    ms_idx = np.where(ms)[0]

    if n == 0:
        ax.text(0.5, 0.5, "N=0", transform=ax.transAxes, ha="center", va="center", fontsize=11)
        ax.set_title(f"{lo}–{hi}°\nN=0")
        ax.set_xlabel(r"$v_\lambda$ (deg/day)")
        continue

    for i in ms_idx:
        ax.plot([matched.vlam.iloc[i], bench.vlam.iloc[i]],
                [matched.vbeta.iloc[i], bench.vbeta.iloc[i]],
                color="gray", lw=0.6, alpha=0.5, zorder=1)

    ax.scatter(bench.vlam.iloc[ms_idx],   bench.vbeta.iloc[ms_idx],
               c="tab:blue", s=45, alpha=0.85, zorder=3, marker="^", label="Benchmark")
    ax.scatter(matched.vlam[ms], matched.vbeta[ms],
               c="tab:red",  s=45, alpha=0.85, zorder=4, label="Sorcha")

    sub_dv = np.hypot(matched.vlam.values[ms_idx] - bench.vlam.values[ms_idx],
                      matched.vbeta.values[ms_idx] - bench.vbeta.values[ms_idx])

    ax.axhline(0, color="k", lw=0.4); ax.axvline(0, color="k", lw=0.4)
    ax.set_title(f"|Δλ☉| = {lo}–{hi}°\nN={n}  median|dv|={np.median(sub_dv):.3f}", fontsize=9)
    ax.set_xlabel(r"$v_\lambda$ (deg/day)", fontsize=10)
    ax.legend(fontsize=7, loc="upper left")
    ax.grid(alpha=0.2)
    ax.set_aspect("equal")

axes[0].set_ylabel(r"$v_\beta$ (deg/day)", fontsize=10)
fig.suptitle("Rate space by direction bin — Sorcha (red) vs benchmark (blue), gray = same object", fontsize=11)
plt.tight_layout()
plt.savefig("Figures/matched_pairs_scatter_by_direction.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved Figures/matched_pairs_scatter_by_direction.png")

## Rate space by direction bin — quiver arrows from origin (5 panels)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 5), sharey=True)

for ax, (lo, hi) in zip(axes, BINS):
    ms = (matched.absdlon >= lo) & (matched.absdlon < hi)
    n = ms.sum()
    ms_idx = np.where(ms)[0]

    if n == 0:
        ax.text(0.5, 0.5, "N=0", transform=ax.transAxes, ha="center", va="center", fontsize=11)
        ax.set_title(f"{lo}–{hi}°\nN=0")
        ax.set_xlabel(r"$v_\lambda$ (deg/day)")
        continue

    zeros_n = np.zeros(n)
    ax.quiver(zeros_n, zeros_n,
              bench.vlam.values[ms_idx], bench.vbeta.values[ms_idx],
              color="tab:blue", alpha=0.55,
              angles="xy", scale_units="xy", scale=1, width=0.005,
              label="Benchmark")
    ax.quiver(zeros_n, zeros_n,
              matched.vlam.values[ms_idx], matched.vbeta.values[ms_idx],
              color="tab:red", alpha=0.55,
              angles="xy", scale_units="xy", scale=1, width=0.005,
              label="Sorcha")

    all_v = np.concatenate([
        matched.vlam.values[ms_idx], bench.vlam.values[ms_idx],
        matched.vbeta.values[ms_idx], bench.vbeta.values[ms_idx]
    ])
    lim = 1.1 * np.abs(all_v).max() if np.abs(all_v).max() > 0 else 1.0
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)

    ax.axhline(0, color="k", lw=0.4); ax.axvline(0, color="k", lw=0.4)
    ax.set_title(f"|Δλ☉| = {lo}–{hi}°\nN={n}", fontsize=9)
    ax.set_xlabel(r"$v_\lambda$ (deg/day)", fontsize=10)
    ax.legend(fontsize=7, loc="upper left")
    ax.grid(alpha=0.2)
    ax.set_aspect("equal")

axes[0].set_ylabel(r"$v_\beta$ (deg/day)", fontsize=10)
fig.suptitle("Velocity vectors from origin by direction bin — Sorcha (red) vs benchmark (blue)", fontsize=11)
plt.tight_layout()
plt.savefig("Figures/matched_pairs_quiver_by_direction.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved Figures/matched_pairs_quiver_by_direction.png")